# Modern Backbone Comparison

This notebook compares the evaluated ResNet50 baseline against modern
compact backbones under a shared Food-101 protocol. Start with frozen
feature extraction for EfficientNet-B0 and ConvNeXt-Tiny, then promote
the best challenger to fine-tuning only if it beats the baseline
tradeoff.

## 1. Comparison Contract

| Backbone | Initial role | Decision signal |
| --- | --- | --- |
| ResNet50 | baseline reference | 73.64% test top-1, 91.18% test top-5 |
| EfficientNet-B0 | efficient challenger | accuracy per parameter and latency |
| ConvNeXt-Tiny | modern high-capacity challenger | top-1/top-5 improvement versus cost |

Use the same train/validation/test split, classifier-head size,
evaluation metrics, and artifact exports as notebook 1.

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for backbone comparison."""

    SEED: int = 42
    BATCH_SIZE: int = 32
    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    NUM_WORKERS: int = 2
    HEAD_EPOCHS: int = 5
    LEARNING_RATE: float = 1e-3
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    RESULTS_DIR: Path = Path("/kaggle/working/results/backbone_comparison")


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the shared Food-101 classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, CFG.NUM_CLASSES),
    )


def freeze_parameters(model: nn.Module) -> None:
    """Freeze all model parameters."""
    for parameter in model.parameters():
        parameter.requires_grad = False


def build_resnet50() -> nn.Module:
    """Build ResNet50 feature-extraction baseline."""
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    freeze_parameters(model)
    model.fc = make_classifier_head(model.fc.in_features)
    return model


def build_efficientnet_b0() -> nn.Module:
    """Build EfficientNet-B0 feature-extraction challenger."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    freeze_parameters(model)
    in_features = model.classifier[1].in_features
    model.classifier = make_classifier_head(in_features)
    return model


def build_convnext_tiny() -> nn.Module:
    """Build ConvNeXt-Tiny feature-extraction challenger."""
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    freeze_parameters(model)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, CFG.NUM_CLASSES)
    return model


MODEL_BUILDERS = {
    "ResNet50": build_resnet50,
    "EfficientNet_B0": build_efficientnet_b0,
    "ConvNeXt_Tiny": build_convnext_tiny,
}

## 2. Next Implementation Step

Add the shared data pipeline and training loop from notebook 1, then run
frozen-head training for each builder in `MODEL_BUILDERS`. Export a
compact comparison table with validation/test top-1, top-5, parameter
count, model size, and latency. Promote only the strongest challenger to
deeper fine-tuning.